# Full experiment — ABX discrimination — configured by one file

> **Full experiment.** Shows a **welcome** screen, then the **consent**
> questionnaire (it sets the anonymous `participant_id`), then the experiment,
> then a **thank-you** screen. Every saved file is named with that participant id.


This notebook runs a **classic ABX** discrimination test and saves the answers.
On each trial the participant hears three intervals — **A**, **B** and **X** —
where **X is a copy of A or B** (chosen at random). They decide whether X matches
A or B. ABX measures how reliably two stimuli can be told apart (chance = 50%).

Like the other whispy experiments it is **config-driven**: the whole thing is
described in [`configs/abx.yml`](../configs/abx.yml); only the shared look comes
from [`configs/design.yml`](../configs/design.yml). You should not need to edit
any Python below.

That one file is read by three consumers:

| block in the YAML | used by | configures |
|---|---|---|
| `output:` + `SoundDevice:` / `OSCHandler:` | the stimuli handler | whether a trial plays a WAV or sends an OSC message per stimulus id |
| `test:` / `ui:` | the ABX window | A/B shuffling, window size, wording |
| `trial:` / `pairs:` / `experiment:` | this notebook | the prompt, the pairs to compare, and how many trials |

Set `output: osc` in `configs/abx.yml` to send OSC messages instead of playing
audio (e.g. to trigger cues in Max/Pd/SuperCollider) — see the `OSCHandler:`
block there. The Python below (`build_stimuli_handler`, the `ABX` window) is
identical either way.

If you haven't installed the project yet, run this from the repo root:
`pip install -e .`

## Welcome the participant

Run this first. It greets the participant with a **welcome screen** before
anything else; its message lives in [`configs/welcome.yml`](../configs/welcome.yml)
(markdown supported), so you can reword it without editing the Python. The cell
blocks until the participant clicks **Continue**.

This cell also opens the **shared experiment window**: every later screen —
consent, trials, thank-you — swaps its content into this one fullscreen
window, so the participant never sees the desktop in between. The thank-you
cell closes it.

This cell also opens the **shared experiment window**: every later screen —
consent, trials, thank-you — swaps its content into this one fullscreen
window, so the participant never sees the desktop in between. The thank-you
cell closes it.

This cell also opens the **shared experiment window**: every later screen —
consent, trials, thank-you — swaps its content into this one fullscreen
window, so the participant never sees the desktop in between. The thank-you
cell closes it.

This cell also opens the **shared experiment window**: every later screen —
consent, trials, thank-you — swaps its content into this one fullscreen
window, so the participant never sees the desktop in between. The thank-you
cell closes it.

In [ ]:
from pathlib import Path
from whispy.ui import ExperimentHost, InfoWindow
from whispy.utils import read_config

# One OS window hosts every screen of this experiment: each UI below is passed
# `parent=host` and swaps its content into that window, so the participant
# never sees the desktop between two screens. The thank-you cell closes it
# (after an error, close it manually with `host.close()`).
host = ExperimentHost()

# The welcome message lives in configs/welcome.yml (markdown supported); reword
# it there without editing this cell.
welcome = read_config(Path('..') / 'configs' / 'welcome.yml')['ui']
InfoWindow(welcome['message'], parent=host, blocking=True)

## Consent

Next, record consent: the cell below shows the consent questionnaire
([`configs/questionnaires/questionnaire_consent.yml`](../configs/questionnaires/questionnaire_consent.yml)),
which records consent and builds an **anonymous participant id** from the
`pid_1..pid_4` answers. The id is kept in the `participant_id` variable and used
to name every result file saved in this notebook, so all of this participant's
data shares one id.

In [ ]:
from pathlib import Path
from whispy.ui import Questionnaire
from whispy.utils import participant_id_from_consent, save_results

consent_config = Path('..') / 'configs' / 'questionnaires' / 'questionnaire_consent.yml'
consent = Questionnaire(questionnaire=consent_config, blocking=True, parent=host)
consent_results = consent.get_results()
# no consent.close() here: the next screen swaps its content into the shared window

# Anonymous participant id (e.g. 'HPo1'); names every result file below.
participant_id = participant_id_from_consent(consent_results)
print('participant id:', participant_id)
save_results(consent_results, 'consent', participant_id=participant_id)

In [3]:
import whispy
import pandas as pd
from pathlib import Path
from whispy.interfaces import build_stimuli_handler
from whispy.ui import ABX
from whispy.utils import read_config

config_path = Path('..') / 'configs' / 'abx.yml'   # <- the one file you edit
stimuli_dir = Path('demo_stimuli/abx')                  # <- folder with your WAVs

cfg = read_config(config_path)
# reads `output:` (sounddevice/osc) and builds the matching handler - swap
# `output: sounddevice` <-> `output: osc` in abx.yml, nothing here changes.
handler = build_stimuli_handler(config_path, stimuli_dir, loop=False)
print('output backend:', cfg.get('output', 'sounddevice'))
print('available stimulus ids:', list(handler.stimuli.keys()))

output backend: sounddevice
available stimulus ids: ['easy_a', 'easy_b', 'hard_a', 'hard_b']


## See your ABX experiment at a glance

Open [`configs/abx.yml`](../configs/abx.yml) to edit every setting (it has inline
comments). The cell below prints the current values and the pairs to be tested,
so you can review the design without leaving the notebook.

In [4]:
print('test      :', cfg['test'])
print('ui        :', cfg['ui'])
print('trial     :', cfg['trial'])
print('experiment:', cfg['experiment'])
print()
print('pairs (each is one A/B comparison):')
for i, p in enumerate(cfg['pairs'], start=1):
    print(f"  {i}. {p.get('name', '')}:  A={p['a']}   B={p['b']}")

test      : {'shuffle_ab': True}
ui        : {'content_area_size': [60, 80], 'listen_label': 'Listen:', 'answer_label': 'Your answer - X is the same as:', 'submit_hint': 'Play A, B and X (keys A/B/X), choose with ←/→, then press Enter to submit. \n You can also click the buttons with the mouse.', 'submit_button_text': 'Submit choice', 'show_progress': True, 'progress_text': 'Trial {current} of {total}'}
trial     : {'task': 'Is **X** the same as **A** or **B**?', 'block_name': 'ABX'}
experiment: {'repetitions': 8, 'shuffle_trials': True, 'seed': 0}

pairs (each is one A/B comparison):
  1. easy (clearly different):  A=easy_a   B=easy_b
  2. hard (subtle):  A=hard_a   B=hard_b


## Using your own stimuli

The WAVs in `examples/demo_stimuli/abx/` are **only a demo** (run
`python examples/demo_stimuli/abx/generate_abx_stimuli.py` to regenerate them). For a
real experiment you swap in your own files — you only touch the config and
`stimuli_dir`, never the Python.

1. **Point `stimuli_dir`** (in the setup cell) at the folder holding your WAVs.
2. **Map ids to files** under `SoundDevice:` in
   [`configs/abx.yml`](../configs/abx.yml), then list the comparisons under
   `pairs:` using those ids:
   ```yaml
   SoundDevice:
     ref:  { file: reference.wav }
     proc: { file: processed.wav }
   pairs:
     - { a: ref, b: proc, name: my comparison }
   ```

**Safety constraints** (checked when the handler loads in the setup cell — a
violation raises a `ValueError`):

- **No clipping** — every sample must satisfy `|amplitude| < 1` (normalise with headroom, e.g. ~0.7).
- **One sampling rate** — every file must share the same sampling rate.
- **All ids must exist** — every id used under `pairs:` must be defined under `SoundDevice:`.

Run the next cell to **pre-flight-check your files before testing a
participant**. After changing the config or `stimuli_dir`, re-run the setup cell.

> The pre-flight check below (`check_stimuli`) validates the `SoundDevice:` WAVs
> and only applies with `output: sounddevice`. With `output: osc` there are no
> files to check — instead make sure every id used under `pairs:` has a matching
> entry under `OSCHandler.stimuli:` in the config.

In [5]:
import numpy as np
import pyfar as pf

def check_stimuli(config_path, stimuli_dir):
    """Validate the stimulus set: existence, clipping, one sampling rate, and
    that every id used by a pair is defined."""
    cfg = read_config(config_path)
    sound = cfg.get('SoundDevice', {})
    pairs = cfg.get('pairs', [])
    folder = Path(stimuli_dir)

    problems, rates = [], {}
    print(f'Checking {len(sound)} stimuli in {folder.resolve()}\n')
    for sid, spec in sound.items():
        path = folder / spec['file']
        if not path.exists():
            problems.append(f"id {sid!r}: file not found -> {path}")
            print(f"  {str(sid):>8}  MISSING   {spec['file']}")
            continue
        sig = pf.io.read_audio(path)
        peak = float(np.max(np.abs(sig.time)))
        rates[sid] = sig.sampling_rate
        flag = '  <-- CLIPPING' if peak >= 1 else ''
        print(f"  {str(sid):>8}  {sig.sampling_rate} Hz  peak={peak:.3f}  {spec['file']}{flag}")
        if peak >= 1:
            problems.append(f"id {sid!r}: clips (peak={peak:.3f} >= 1)")

    for pair in pairs:
        for role in ('a', 'b'):
            rid = pair.get(role)
            if rid not in sound:
                problems.append(f"pair {pair.get('name', pair)!r} uses id {rid!r}, not defined under SoundDevice:")

    if len(set(rates.values())) > 1:
        problems.append(f"mixed sampling rates: {sorted(set(rates.values()))} (all must match)")

    print()
    if problems:
        print('NOT ready - fix these before running:')
        for prob in problems:
            print('  -', prob)
    else:
        print('All good: files exist, no clipping, one sampling rate, all pair ids defined.')
    return not problems

check_stimuli(config_path, stimuli_dir)

Checking 4 stimuli in D:\Uni\Sem IV\P&A\Whispy\whispy\examples\demo_stimuli\abx

    easy_a  44100 Hz  peak=0.700  easy_a.wav
    easy_b  44100 Hz  peak=0.700  easy_b.wav
    hard_a  44100 Hz  peak=0.700  hard_a.wav
    hard_b  44100 Hz  peak=0.700  hard_b.wav

All good: files exist, no clipping, one sampling rate, all pair ids defined.


True

## Build the trial list

Each pair is presented `experiment.repetitions` times. For every trial we also
decide here (with the config's `seed`, so it is reproducible) whether **X** is a
copy of A or B. The ABX window additionally randomizes which stimulus is shown as
A vs B when `test.shuffle_ab` is on.

In [6]:
import random

exp = cfg['experiment']
trial = cfg['trial']
rng = random.Random(exp.get('seed', 0))

trials = []
for pair in cfg['pairs']:
    for _ in range(int(exp.get('repetitions', 1))):
        t = dict(pair)
        t['x'] = rng.choice([pair['a'], pair['b']])   # which stimulus X copies
        trials.append(t)
if exp.get('shuffle_trials', True):
    rng.shuffle(trials)

def build_screen(pair, trial_index):
    return {
        'a': pair['a'], 'b': pair['b'], 'x': pair['x'],
        'task': trial['task'],
        'block': 0, 'section': 0,
        'trial_id': trial_index,
        'block_name': trial.get('block_name', 'ABX'),
        'section_name': pair.get('name', 'pair'),
        # drives the "Trial X of Y" progress bar (ui.show_progress)
        'progress': {'current': trial_index, 'total': len(trials)},
    }

print(f"{len(trials)} trials ({len(cfg['pairs'])} pairs x {exp.get('repetitions')} repetitions)")

16 trials (2 pairs x 8 repetitions)


## Run it

The loop below presents every trial inside the **shared experiment window**
(`parent=host`): each trial swaps its content into the window opened by the
welcome cell, so it stays fullscreen with no reload. The window stays open
afterwards — the thank-you cell closes it.

Every completed trial is **autosaved** to this run's results CSV right away,
so even a crash mid-run can only lose the trial in progress.

In each trial: press **A**, **B**, **X** (or click) to hear each interval, choose
the match with **←/→** (or click A/B), then press **Enter** to submit.

In [ ]:
from whispy.utils import ResultsAutosaver

# Crash-safe logging: the accumulated results are rewritten to one CSV after
# every trial, so a dying kernel loses at most the trial in progress. The
# file uses the usual naming scheme and already IS this run's results file.
autosaver = ResultsAutosaver('abx', participant_id=globals().get('participant_id'))

def run_trial(screen):
    abx = ABX(screen=screen, stimuli_handler=handler,
              abx_config=config_path, blocking=True, parent=host)
    return abx.get_results()

rows = []
try:
    for i, pair in enumerate(trials, start=1):
        rows.append(run_trial(build_screen(pair, i)))
        autosaver.save(pd.concat(rows, ignore_index=True))
except BaseException:
    host.close()   # don't leave the fullscreen window stuck after an error
    raise

results = pd.concat(rows, ignore_index=True)
results

## Thank the participant

The experiment is over. Run this cell to show the participant a **thank-you
screen**; its message lives in [`configs/thanks.yml`](../configs/thanks.yml)
(markdown supported), so you can reword it without editing the Python.

In [ ]:
from whispy.ui import InfoWindow

# The thank-you message lives in configs/thanks.yml (markdown supported); reword
# it there without editing this cell.
thanks = read_config(Path('..') / 'configs' / 'thanks.yml')['ui']
InfoWindow(thanks['message'], parent=host, blocking=True)

# the experiment is over - close the shared experiment window
host.close()

## Results & percent correct

`results` has one row per trial (`a`, `b`, `x`, `correct`, `selected`,
`correct_bool`, `rt`, ...). ABX performance is summarized as **percent correct**,
overall and per pair. Chance is **50%**; well above that means the pair is
discriminable.

In [9]:
overall = results['correct_bool'].mean()
print(f'overall percent correct: {overall * 100:.1f}%   (n={len(results)}, chance=50%)')
print()
per_pair = (results.groupby('section_name')['correct_bool']
            .agg(percent_correct=lambda s: round(s.mean() * 100, 1), n='count'))
print('per pair:')
print(per_pair.to_string())

overall percent correct: 81.2%   (n=16, chance=50%)

per pair:
                          percent_correct  n
section_name                                
easy (clearly different)             87.5  8
hard (subtle)                        75.0  8


## Save the results

The results are **already on disk**: the run cell rewrites its CSV (in a
`results/` folder created next to this notebook) after every trial, so even a
kernel that dies mid-run loses at most the trial in progress. The file name
always carries a **timestamp**. If a `participant_id` is in scope (set by a
consent block earlier in the notebook) it is included
(`<name>_<id>_<timestamp>.csv`); otherwise an iterating fallback number is
used (`<name>_<NNN>_<timestamp>.csv`). Existing files are never overwritten.
The cell below just reports the file.

In [ ]:
# The run cell above already saved the results incrementally after every
# trial (crash-safe), so there is nothing left to write - just report the file.
results_path = autosaver.path
print('saved results to', results_path.resolve())

## Plot the results

Plot up to three plots. A plot with the accuracy by section can be created. It will also show the 95% confidence interval.
A boxplot of the reaction times by correctness can be created as well and lastly, the reaction time and correctness over the trials can be illustrated by graphs.

In [ ]:
from whispy.utils import Plotting

plotting = Plotting()

plotting.plot_abx_accuracy_by_section(results)
plotting.plot_abx_rt_boxplot(results)
plotting.plot_abx_correctness_rt_over_trials(results)